# Load Dataset from HF

This will gather the dataset created using the dataset and load it into memory.


In [ ]:
# lib import
import os
import numpy as np
from datasets import load_dataset, get_dataset_config_names

# setup
data = {}
eval_dir = os.path.join(os.getcwd(), "eval")
os.makedirs(eval_dir, exist_ok=True)
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "wikipos")
configs = get_dataset_config_names("whatphiliptrains/wikipos")

# subset configuration
subset_size = None
rng = np.random.default_rng(97)

for config in configs:
    dataset = load_dataset(
        "whatphiliptrains/wikipos", config, cache_dir=source_ds_cache_dir
    )
    if subset_size is not None and len(dataset["train"]) > subset_size:
        indices = rng.choice(len(dataset["train"]), size=subset_size, replace=False)
        dataset["train"] = dataset["train"].select(indices)
        print(
            f"[{config}] Subset selected: {subset_size} samples from {len(dataset['train'])} total"
        )

    data[config] = dataset


# KNN and CPD
based on 'DREAMS' (Kury et al.) and 'The art of using t-SNE for single-cell transcriptomics' (Kobak and Berens) quality measurements; Correlation of Pairwise Distances (CPD) as global structure quality indicator, The k-nearest neighbor recall (KNN) average (Mean Nearest Neighbor implementation) to evaluate local neighborhoods. To run KNN over entire dataset multithread processing is used, potentially introducing slight non-deterministic fluctuations.

In [ ]:
import csv
import scipy
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import pdist
from datetime import datetime
from tqdm import tqdm

n = 10
results = {}


def embedding_quality_unlabeled(X, Z, knn=10, subsetsize=1000, seed=None):
    # local structure preservation (mnn)
    nbrs1 = NearestNeighbors(n_neighbors=knn, n_jobs=-1).fit(X)  # non-deterministic
    ind1 = nbrs1.kneighbors(return_distance=False)

    nbrs2 = NearestNeighbors(n_neighbors=knn, n_jobs=-1).fit(Z)  # non-deterministic
    ind2 = nbrs2.kneighbors(return_distance=False)

    intersections = 0.0
    for i in range(X.shape[0]):
        intersections += len(set(ind1[i]) & set(ind2[i]))
    mnn = intersections / X.shape[0] / knn

    # distance correlation (rho)
    rng = np.random.default_rng(seed)
    subset = rng.choice(X.shape[0], size=subsetsize, replace=False)

    d1 = pdist(X[subset, :])
    d2 = pdist(Z[subset, :])
    rho = scipy.stats.spearmanr(d1, d2).correlation

    return (mnn, rho)


for ds in tqdm(data, desc="Calculating quality metrics..."):
    current = data[ds]["train"]
    pos = np.column_stack([current["x"], current["y"]])
    embeddings = np.array(current["embeddings"])

    knn, cpd = embedding_quality_unlabeled(X=embeddings, Z=pos, knn=n, seed=97)

    results[ds] = {"knn": knn, "cpd": cpd}

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filepath = os.path.join(eval_dir, f"eval-{str(subset_size)}-{timestamp}.csv")
with open(filepath, "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["config", "knn", "cpd"])
    for config, metrics in results.items():
        writer.writerow([config, metrics["knn"], metrics["cpd"]])

print(f"Results written to '{filepath}'")